### Import

In [1]:
import re
import os
import gc
import html
import json
import string
import inflect
import warnings
import unicodedata
import contractions

import numpy as np
import pandas as pd
import datetime as dt
from tqdm.auto import tqdm

import seaborn as sns
import matplotlib.pyplot as plot

from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import DataLoader

import evaluate
import datasets

from datasets import Dataset
from datasets import DatasetDict
from datasets import load_dataset
from datasets import list_metrics

from transformers import AdamW
from transformers import get_scheduler

from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification

pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = 10000
pd.set_option('display.max_columns', 500)

### General Path

In [2]:
path_data = "../Data/EHR/"
path_out  = '../Data/TEXT/'
mimiciii = "PATH TO DATA/mimic-iii(1.4)/"
abbreviation_path = '../Data/Abbreviations/Abbreviation.txt'

### Read Data

In [3]:
all_note_ids = pd.read_csv(path_data + 'all_note_ids.csv', low_memory=False, index_col=False)
all_note_ids = all_note_ids.rename(columns={"Note": "ROW_ID"})
all_note_ids.head(2)

In [4]:
print(all_note_ids.ICUSTAY_ID.nunique())
print(all_note_ids.shape)

56520
(1161218, 9)


### Prepare Note IDs in EHR Records

In [5]:
list_of_all_patients = all_note_ids.groupby('ICUSTAY_ID')[['ICUSTAY_ID', 'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG']].head(1)
list_of_all_patients = list_of_all_patients.reset_index(drop=True)
list_of_all_patients.head(2)

### Read Clinical Notes

In [6]:
note = pd.read_csv(mimiciii + "NOTEEVENTS.csv")
note = note[['SUBJECT_ID', 'HADM_ID', 'CHARTDATE', 'CHARTTIME', 'CATEGORY', 'ROW_ID', 'TEXT']]

<ipython-input>:1: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  note = pd.read_csv(mimiciii + "NOTEEVENTS.csv")


In [7]:
reports = note[note.CATEGORY != 'Discharge summary'].copy()
discharges = note[note.CATEGORY == 'Discharge summary'].copy()

In [8]:
reports['CHARTDATE'] = pd.to_datetime(reports['CHARTDATE'])
reports['CHARTTIME'] = pd.to_datetime(reports['CHARTTIME'])
discharges['CHARTDATE'] = pd.to_datetime(discharges['CHARTDATE'])
discharges['CHARTTIME'] = pd.to_datetime(discharges['CHARTTIME'])

condition_1 = (reports['CHARTTIME'].isnull())
condition_2 = (discharges['CHARTTIME'].isnull())
reports.loc[condition_1, 'CHARTTIME'] = reports.loc[condition_1, 'CHARTDATE']
discharges.loc[condition_2, 'CHARTTIME'] = discharges.loc[condition_2, 'CHARTDATE']

reports.drop(columns=['CHARTDATE'], inplace=True)
discharges.drop(columns=['CHARTDATE'], inplace=True)

reports = reports.reset_index(drop=True)
discharges = discharges.reset_index(drop=True)

<ipython-input>:9: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  discharges.loc[condition_2, 'CHARTTIME'] = discharges.loc[condition_2, 'CHARTDATE']


In [9]:
del note
del discharges
gc.collect()

100

### Final Reports

In [10]:
reports.head(1)

In [11]:
print("# table size = ", reports.shape)
print("# unique note = ", reports.ROW_ID.nunique())
print("# unique patients = ", reports.SUBJECT_ID.nunique())
print("# unique hospital admissions = ", reports.HADM_ID.nunique())
print("# note types = ", reports.CATEGORY.unique())

# table size =  (2023528, 6)
# unique note =  2023528
# unique patients =  46047
# unique hospital admissions =  58028
# note types =  ['Echo' 'ECG' 'Nursing' 'Physician ' 'Rehab Services' 'Case Management '
 'Respiratory ' 'Nutrition' 'General' 'Social Work' 'Pharmacy' 'Consult'
 'Radiology' 'Nursing/other']


### Filter Reports Based on ICU-Stay

In [12]:
all_reports = all_note_ids.merge(reports, on=['SUBJECT_ID', 'HADM_ID', 'ROW_ID'], how='left')

In [13]:
all_reports['INTIME']  = pd.to_datetime(all_reports['INTIME'])
all_reports['OUTTIME'] = pd.to_datetime(all_reports['OUTTIME'])
all_reports['CHARTTIME'] = pd.to_datetime(all_reports['CHARTTIME'])

In [14]:
all_reports = all_reports[((all_reports['CHARTTIME'] >= all_reports['INTIME']) & (all_reports['CHARTTIME'] <= all_reports['OUTTIME']))]

In [15]:
all_reports = all_reports[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID',
                           'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG',
                           'CATEGORY', 'ROW_ID', 'TEXT']]

all_reports['ICU_EXPIRE_FLAG'] = all_reports['ICU_EXPIRE_FLAG'].astype(int)
all_reports['HOSPITAL_EXPIRE_FLAG'] = all_reports['HOSPITAL_EXPIRE_FLAG'].astype(int)

all_reports = all_reports.drop_duplicates()

all_reports = all_reports.reset_index(drop=True)

In [16]:
all_reports.head(1)

In [17]:
print("# table size = ", all_reports.shape)
print("# unique note = ", all_reports.ROW_ID.nunique())
print("# unique patients = ", all_reports.SUBJECT_ID.nunique())
print("# unique hospital admissions = ", all_reports.HADM_ID.nunique())
print("# unique icu admissions = ", all_reports.ICUSTAY_ID.nunique())
print("# note types = ", all_reports.CATEGORY.unique())

# table size =  (1159443, 8)
# unique note =  1159443
# unique patients =  43458
# unique hospital admissions =  53247
# unique icu admissions =  56515
# note types =  ['Radiology' 'Nursing/other' 'ECG' 'Echo' 'Physician ' 'Nursing'
 'Nutrition' 'Case Management ' 'Rehab Services' 'Social Work'
 'Respiratory ' 'General' 'Pharmacy' 'Consult']


### Convert Data to Dataset

In [18]:
df = Dataset.from_pandas(all_reports)

In [19]:
df

Dataset({
    features: ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'CATEGORY', 'ROW_ID', 'TEXT'],
    num_rows: 1159443
})

### Remove Patterns

In [20]:
def pattern_repl(matchobj):

    return ' '.rjust(len(matchobj.group(0)))

def find_end(text):
    
    ends = [len(text)]
    patterns = [
        re.compile(r'BY ELECTRONICALLY SIGNING THIS REPORT', re.I),
        re.compile(r'\n {3,}DR.', re.I),
        re.compile(r'[ ]{1,}RADLINE ', re.I),
        re.compile(r'.*electronically signed on', re.I),
        re.compile(r'M\[0KM\[0KM')]
    
    for pattern in patterns:
        matchobj = pattern.search(text)
        if matchobj:
            ends.append(matchobj.start())
    return min(ends)

def remove_pattern(text):

    text = re.sub(r'\[\*\*.*?\*\*\]', pattern_repl, text)
    text = re.sub(r'_', ' ', text)

    start = 0
    end = find_end(text)
    new_text = ''
    if start > 0:
        new_text += ' ' * start
    new_text = text[start:end]

    if len(text) - end > 0:
        new_text += ' ' * (len(text) - end)
    return new_text

### Filter based on Titles

In [21]:
SECTION_TITLES = re.compile(
                r'(ABDOMEN AND PELVIS|CLINICAL HISTORY|CLINICAL INDICATION|COMPARISON|COMPARISON STUDY DATE'
                r'|EXAM|EXAMINATION|FINDINGS|HISTORY|IMPRESSION|INDICATION|TRANSFER NOTE'
                r'|MEDICAL CONDITION|PROCEDURE|REASON FOR EXAM|REASON FOR STUDY|REASON FOR THIS EXAMINATION'
                r'|PORTABLE CHEST|ASSESSMENT|INTERPRETATION|CONCLUSIONS|REASON|ADMITTING DIAGNOSIS|STUDY|WET READ|NURSING ACCEPTANCE|ASSESS|PLAN'
                r'|TECHNIQUE'  
                r'|NEURO|CV|GI|GU|GI/GU|SKIN|IVF|RESP|ENOD|A/P'
                r'):|FINAL REPORT',
                re.I | re.M)

def has_meaningful_content(content):
    
    return bool(re.search(r'\w+', content))

def split_and_concatenate_title(text):
    
    sections = []
    matches = list(SECTION_TITLES.finditer(text))
    
    if not matches:
        return text
    else:
        for i, match in enumerate(matches):
            title = match.group().strip()
            content_start = match.end()
            content_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            content = text[content_start:content_end].strip()

            if has_meaningful_content(content):
                if sections and not text[matches[i - 1].end():match.start()].strip():
                    sections[-1] += "\n" + title + "\n" + content
                else:
                    sections.append(title + "\n" + content)
        return "\n".join(sections)

### Raw Text Length

In [22]:
def compute_text_length(text):
    
    len_text = len(text.split())
    
    return len_text

### Remove Extra Space 

In [23]:
def remove_wspaceA(text):
    
    clean_text = text.strip()
    clean_text = " ".join(clean_text.split())
    
    return clean_text

def remove_wspaceB(text):
    
    clean_text = re.sub('\s+', ' ', text)
    
    return clean_text

### Accented Characters to ASCII

In [24]:
def ascii_convert(text):
    
    clean_text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    
    return clean_text

### Decontraction

In [25]:
def decontractionA(text):
    
    clean_text = contractions.fix(text)
    
    return clean_text

In [26]:
def decontractionB(phrase):

    phrase = re.sub(r"won\'t", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not",  phrase)
    phrase = re.sub(r"won\’t", "will not", phrase)
    phrase = re.sub(r"can\’t", "can not",  phrase)
    phrase = re.sub(r"n\'t", " not",  phrase)
    phrase = re.sub(r"\'re", " are",  phrase)
    phrase = re.sub(r"\'s" , " is",   phrase)
    phrase = re.sub(r"\'d" , " would",phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t" , " not",  phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m" , " am",   phrase)
    phrase = re.sub(r"n\’t", " not",  phrase)
    phrase = re.sub(r"\’re", " are",  phrase)
    phrase = re.sub(r"\’s" , " is",   phrase)
    phrase = re.sub(r"\’d" , " would",phrase)
    phrase = re.sub(r"\’ll", " will", phrase)
    phrase = re.sub(r"\’t" , " not",  phrase)
    phrase = re.sub(r"\’ve", " have", phrase)
    phrase = re.sub(r"\’m" , " am",   phrase)

    return phrase

### Clinical Abbreviation

In [27]:
with open(abbreviation_path, 'r') as file:
    lines = file.readlines()
    
abbreviations = {}

for line in lines:
    parts = line.strip().split(' ', 1)
    abbr, full_form = parts
    if len(abbr) >= 3:  
        abbreviations[abbr] = full_form.strip()

In [28]:
def expand_abbreviations(text):
    
    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in abbreviations.keys()) + r')\b')
    clean_text  = pattern.sub(lambda x: abbreviations[x.group()], text)
    
    return clean_text

### Remove Punctuation

In [29]:
def replace_punc_with_space(text):
    
    mypunctuation = '!"#%&\'*+,/;<=>?@\\^_`|~'
    
    translation_table = str.maketrans(mypunctuation, ' ' * len(mypunctuation))
    clean_text = text.translate(translation_table)
    
    return clean_text

### Sentence & Word Count

In [30]:
def get_word_count_text(text):
    
    sent_count = 0
    word_count = 0
    vocab = {}
    
    sentences = sent_tokenize(str(text).lower())
    sent_count = sent_count + len(sentences)
    
    for sentence in sentences:
        words = word_tokenize(sentence)
        for word in words:
            if(word in vocab.keys()):
                vocab[word] = vocab[word] +1
            else:
                vocab[word] =1 
    word_count = len(vocab.keys())
    
    return word_count

In [31]:
def get_sentence_count_text(text):
    
    sent_count = 0
    word_count = 0
    vocab = {}
    
    sentences = sent_tokenize(str(text).lower())
    sent_count = sent_count + len(sentences)
    
    for sentence in sentences:
        words = word_tokenize(sentence)
        for word in words:
            if(word in vocab.keys()):
                vocab[word] = vocab[word] +1
            else:
                vocab[word] =1 
    word_count = len(vocab.keys())
    
    return sent_count

### Apply Procesing Functions

In [ ]:
df = df.map(lambda x: {"CLEAN_TEXT": remove_pattern(x["TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": remove_wspaceA(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": remove_wspaceB(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": ascii_convert(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": [html.unescape(o) for o in x["CLEAN_TEXT"]]}, batched=True)
df = df.map(lambda x: {"CLEAN_TEXT": split_and_concatenate_title(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": decontractionA(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": decontractionB(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": expand_abbreviations(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": replace_punc_with_space(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": remove_wspaceA(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"CLEAN_TEXT": remove_wspaceB(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"n_words_raw": get_word_count_text(x["TEXT"])})
df = df.map(lambda x: {"n_sents_raw": get_sentence_count_text(x["TEXT"])})
df = df.map(lambda x: {"len_text_raw": compute_text_length(x["TEXT"])})
df = df.map(lambda x: {"n_words_clean": get_word_count_text(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"n_sents_clean": get_sentence_count_text(x["CLEAN_TEXT"])})
df = df.map(lambda x: {"len_text_clean": compute_text_length(x["CLEAN_TEXT"])})

### Check Clean Examples

In [33]:
df[0]

### Select Columns 

In [34]:
selected_columns_raw   = ['ICUSTAY_ID', 'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'CATEGORY', 'ROW_ID', 'TEXT', 'len_text_raw']
selected_columns_clean = ['ICUSTAY_ID', 'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'CATEGORY', 'ROW_ID', 'CLEAN_TEXT', 'len_text_clean']

raw_dataset   = df.remove_columns([col for col in df.column_names if col not in selected_columns_raw])
clean_dataset = df.remove_columns([col for col in df.column_names if col not in selected_columns_clean])

clean_dataset = clean_dataset.rename_column("CLEAN_TEXT", "TEXT")

### Filter Short Text

In [35]:
raw_min_length = 15
clean_min_length = 10

raw_dataset = raw_dataset.filter(lambda x: x["len_text_raw"] > raw_min_length)
clean_dataset = clean_dataset.filter(lambda x: x["len_text_clean"] > clean_min_length)

raw_dataset = raw_dataset.remove_columns(['len_text_raw'])
clean_dataset = clean_dataset.remove_columns(['len_text_clean'])

Filter:   0%|          | 0/1159443 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1159443 [00:00<?, ? examples/s]

In [36]:
raw_dataset

Dataset({
    features: ['ICUSTAY_ID', 'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'CATEGORY', 'ROW_ID', 'TEXT'],
    num_rows: 1137088
})

In [37]:
clean_dataset

Dataset({
    features: ['ICUSTAY_ID', 'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'CATEGORY', 'ROW_ID', 'TEXT'],
    num_rows: 1133816
})

### Add Indicator of having original Note

In [38]:
# raw_new_column   = [1] * len(raw_dataset)
# clean_new_column = [1] * len(clean_dataset)

# raw_dataset = raw_dataset.add_column("hasNotes", raw_new_column)
# clean_dataset = clean_dataset.add_column("hasNotes", clean_new_column)

### Add Missing Notes

In [39]:
def add_missing_notes(list_of_all_patients, dataset, target_column):
    
    tmp_df = dataset.to_pandas()
    raw_df = tmp_df.copy()
    
    new_rows = []
    
    
    for index, row in list_of_all_patients.iterrows():
        icustay_id = row['ICUSTAY_ID']
        
        if icustay_id not in raw_df['ICUSTAY_ID'].values:
            
            new_row = {'ICUSTAY_ID': icustay_id,
                       'ICU_EXPIRE_FLAG': int(row['ICU_EXPIRE_FLAG']),
                       'HOSPITAL_EXPIRE_FLAG': int(row['HOSPITAL_EXPIRE_FLAG']),
                       'hasNotes': int(0),
                       'CATEGORY': 'Synthetic',
                       'ROW_ID': np.random.randint(100000, 200000), 
                       target_column : "No clinical text note is available for this patient's record."}
            
            new_rows.append(new_row)
    
    new_rows_df = pd.DataFrame(new_rows)
    updated_df = pd.concat([raw_df, new_rows_df], ignore_index=True)
    updated_dataset = Dataset.from_pandas(updated_df)
    
    return updated_dataset

In [40]:
# raw_dataset_all_icu   = add_missing_notes(list_of_all_patients, raw_dataset,   'TEXT')
# clean_dataset_all_icu = add_missing_notes(list_of_all_patients, clean_dataset, 'TEXT')

### Add Phrase

In [41]:
def add_phrase(text):
    
    phrase = "This is a clinical note from an ICU stay aimed at assessing the risk of mortality:\n"
    clean_text = phrase + text
    
    return clean_text

In [42]:
# clean_dataset_all_icu = clean_dataset_all_icu.map(lambda x: {"TEXT": add_phrase(x["TEXT"])})

### Save Datasets

In [43]:
raw_dataset.save_to_disk(path_out + "MIMICIII_RAW_NOTES")
clean_dataset.save_to_disk(path_out + "MIMICIII_CLEAN_NOTES")

# raw_dataset_all_icu.save_to_disk(path_out + "MIMICIII_RAW_NOTES_ALL_ICU_PROMT")
# clean_dataset_all_icu.save_to_disk(path_out + "MIMICIII_CLEAN_NOTES_ALL_ICU_PROMT")

Saving the dataset (0/4 shards):   0%|          | 0/1137088 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/1133816 [00:00<?, ? examples/s]